# Student Initialisation
- Approach A: random weights
- Approach B: Take a Larger model and cut it down
  1. Layer based pruning
  2. Width based pruning

### Layer Pruning
____
- Idea is to remove some of layers
- Typically remove the middle layer not the last layers.




### Width Pruning
___
- remove the internal dims of the model

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
from huggingface_hub import login
hf_token = ""
login(token=hf_token)

In [ ]:
teacher_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)
teacher_model = AutoModelForMaskedLM.from_pretrained(teacher_name).to(device)
print(f"Teacher model loaded on {device}.")

In [ ]:
class MakingDatasetForStudentModel:
    def __init__(self, tokenizer, model, batch_size=8):
        self.tokenizer = tokenizer
        self.masked_token_id = self.tokenizer.mask_token_id
        self.special_tokens = self.tokenizer.all_special_ids
        self.model = model
        self.batch_size = batch_size
        self.model.eval()
        self.device = next(self.model.parameters()).device

    def __call__(self, texts, path_to_save):
        all_input_ids = []
        all_attention_masks = []
        all_labels = []
        all_teacher_logits = []
        all_masked_indices = []

        special_tokens_tensor = torch.tensor(self.special_tokens, device=self.device)

        for start in range(0, len(texts), self.batch_size):
            batch_texts = texts[start : start + self.batch_size]

            inputs = self.tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True).to(self.device)
            labels = inputs.input_ids.clone()

            special_token_mask = torch.isin(inputs.input_ids, special_tokens_tensor)
            probability_matrix = torch.full(inputs.input_ids.shape, 0.15, device=self.device)
            probability_matrix.masked_fill_(special_token_mask, 0.0)

            masked_indices = torch.bernoulli(probability_matrix).bool()
            labels[~masked_indices] = -100
            inputs.input_ids[masked_indices] = self.masked_token_id

            with torch.no_grad():
                outputs = self.model(**inputs)
                teacher_logits = outputs.logits[masked_indices]

            all_input_ids.append(inputs.input_ids.cpu())
            all_attention_masks.append(inputs.attention_mask.cpu())
            all_labels.append(labels.cpu())
            all_teacher_logits.append(teacher_logits.cpu())
            all_masked_indices.append(masked_indices.cpu())

            del inputs, labels, masked_indices, outputs, teacher_logits
            torch.cuda.empty_cache()

            batch_num = start // self.batch_size + 1
            total_batches = (len(texts) + self.batch_size - 1) // self.batch_size
            processed = min(start + self.batch_size, len(texts))
            print(f"\r  Batch {batch_num}/{total_batches}: processed {processed}/{len(texts)} sentences", end="", flush=True)

        print()

        dataset_to_save = {
            "input_ids": all_input_ids,
            "attention_mask": all_attention_masks,
            "labels": all_labels,
            "teachers_logits": all_teacher_logits,
            "masked_indices": all_masked_indices,
        }

        os.makedirs(os.path.dirname(path_to_save), exist_ok=True)
        torch.save(dataset_to_save, path_to_save)
        print(f"Dataset successfully saved to: {path_to_save}")

In [ ]:
dataset_maker = MakingDatasetForStudentModel(tokenizer, teacher_model)

print("Downloading real dataset...")
wiki_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
real_texts = [text for text in wiki_dataset['train']['text'] if len(text.strip()) > 50]
print(f"Total valid sentences found: {len(real_texts)}")

subset_texts = real_texts[:1000]
save_path = os.path.join(os.getcwd(), "data", "real_wikitext_data.pt")

print(f"\nExtracting Teacher logits for {len(subset_texts)} sentences...")
dataset_maker(texts=subset_texts, path_to_save=save_path)

In [ ]:
del teacher_model, dataset_maker
torch.cuda.empty_cache()
print("Teacher model freed from GPU.")

dataset = torch.load(os.path.join(os.getcwd(), "data", "real_wikitext_data.pt"), map_location="cpu")

student_model = AutoModelForMaskedLM.from_pretrained("google/mobilebert-uncased").to(device)
optimizer = torch.optim.AdamW(student_model.parameters(), lr=5e-5)

kld_loss_fn = nn.KLDivLoss(reduction="batchmean")
ce_loss_fn = nn.CrossEntropyLoss()

T = 2.0
alpha = 0.5
epochs = 3
num_batches = len(dataset["input_ids"])

for epoch in range(epochs):
    student_model.train()
    epoch_loss = 0.0

    for i in range(num_batches):
        input_ids = dataset["input_ids"][i].to(device)
        attention_mask = dataset["attention_mask"][i].to(device)
        masked_indices = dataset["masked_indices"][i].to(device)
        teacher_logits = dataset["teachers_logits"][i].to(device)
        labels = dataset["labels"][i].to(device)

        student_output = student_model(input_ids=input_ids, attention_mask=attention_mask)
        student_logits = student_output.logits[masked_indices]

        student_log_probs = F.log_softmax(student_logits / T, dim=-1)
        teacher_probs = F.softmax(teacher_logits / T, dim=-1)
        distillation_loss = kld_loss_fn(student_log_probs, teacher_probs) * (T ** 2)

        hard_labels = labels[masked_indices]
        hard_loss = ce_loss_fn(student_logits, hard_labels)

        loss = alpha * distillation_loss + (1 - alpha) * hard_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        if (i + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs} | Batch {i+1}/{num_batches} | Loss: {loss.item():.4f}", end="\r")

    avg_loss = epoch_loss / num_batches
    print(f"  Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}                    ")

print("\nTraining complete!")
student_model.save_pretrained(os.path.join(os.getcwd(), "data", "distilled_student_model"))
print("Student model saved.")

In [ ]:
model_path = os.path.join(os.getcwd(), "data", "distilled_student_model")
student_model = AutoModelForMaskedLM.from_pretrained(model_path).to(device)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
student_model.eval()

def predict_masked(text, top_k=5):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    mask_positions = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

    with torch.no_grad():
        logits = student_model(**inputs).logits

    for pos in mask_positions:
        probs = F.softmax(logits[0, pos], dim=-1)
        top_probs, top_ids = probs.topk(top_k)
        top_tokens = tokenizer.convert_ids_to_tokens(top_ids)

        print(f"\nPosition {pos.item()}: '{text.split()[pos.item() - 1]}' predictions:")
        for token, prob in zip(top_tokens, top_probs):
            print(f"  {token:15s} {prob.item():.4f}")

    best_ids = logits[0, mask_positions].argmax(dim=-1)
    filled = inputs.input_ids.clone()
    filled[0, mask_positions] = best_ids
    return tokenizer.decode(filled[0], skip_special_tokens=True)

test_sentences = [
    "The capital of France is [MASK].",
    "She went to the [MASK] to buy some groceries.",
    "Deep learning is a subset of [MASK] learning.",
    "The capital of INDIA is [MASK]."
]

for sentence in test_sentences:
    print(f"\nInput:  {sentence}")
    result = predict_masked(sentence)
    print(f"Output: {result}")

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
from huggingface_hub import login
hf_token = ""
login(token=hf_token)

In [ ]:
teacher_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)
teacher_model = AutoModelForMaskedLM.from_pretrained(teacher_name).to(device)
print(f"Teacher model loaded on {device}.")

In [ ]:
class MakingDatasetForStudentModel:
    def __init__(self, tokenizer, model, batch_size=32):
        self.tokenizer = tokenizer
        self.masked_token_id = self.tokenizer.mask_token_id
        self.special_tokens = self.tokenizer.all_special_ids
        self.model = model
        self.batch_size = batch_size
        self.model.eval()
        self.device = next(self.model.parameters()).device

    def __call__(self, texts, save_dir):
        os.makedirs(save_dir, exist_ok=True)
        special_tokens_tensor = torch.tensor(self.special_tokens, device=self.device)
        total_batches = (len(texts) + self.batch_size - 1) // self.batch_size

        for start in range(0, len(texts), self.batch_size):
            batch_texts = texts[start : start + self.batch_size]

            inputs = self.tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True).to(self.device)
            labels = inputs.input_ids.clone()

            special_token_mask = torch.isin(inputs.input_ids, special_tokens_tensor)
            probability_matrix = torch.full(inputs.input_ids.shape, 0.15, device=self.device)
            probability_matrix.masked_fill_(special_token_mask, 0.0)

            masked_indices = torch.bernoulli(probability_matrix).bool()
            labels[~masked_indices] = -100
            inputs.input_ids[masked_indices] = self.masked_token_id

            with torch.no_grad():
                outputs = self.model(**inputs)
                teacher_logits = outputs.logits[masked_indices]

            batch_data = {
                "input_ids": inputs.input_ids.cpu(),
                "attention_mask": inputs.attention_mask.cpu(),
                "labels": labels.cpu(),
                "teachers_logits": teacher_logits.cpu(),
                "masked_indices": masked_indices.cpu(),
            }

            batch_num = start // self.batch_size
            torch.save(batch_data, os.path.join(save_dir, f"batch_{batch_num:04d}.pt"))

            del inputs, labels, masked_indices, outputs, teacher_logits, batch_data
            torch.cuda.empty_cache()

            print(f"\r  Batch {batch_num + 1}/{total_batches}: processed {min(start + self.batch_size, len(texts))}/{len(texts)} sentences", end="", flush=True)

        print(f"\n{total_batches} batches saved to: {save_dir}")

In [ ]:
dataset_maker = MakingDatasetForStudentModel(tokenizer, teacher_model)

print("Downloading real dataset...")
wiki_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
real_texts = [text for text in wiki_dataset['train']['text'] if len(text.strip()) > 50]
print(f"Total valid sentences found: {len(real_texts)}")

subset_texts = real_texts
data_dir = os.path.join(os.getcwd(), "data", "batches")

print(f"\nExtracting Teacher logits for {len(subset_texts)} sentences...")
dataset_maker(texts=subset_texts, save_dir=data_dir)

In [ ]:
from transformers import BertConfig, BertForMaskedLM

teacher_params = sum(p.numel() for p in teacher_model.parameters())
teacher_config = teacher_model.config

student_config = BertConfig(
    vocab_size=teacher_config.vocab_size,
    hidden_size=teacher_config.hidden_size,
    num_hidden_layers=teacher_config.num_hidden_layers // 2,
    num_attention_heads=teacher_config.num_attention_heads,
    intermediate_size=teacher_config.intermediate_size,
)

student_model = BertForMaskedLM(student_config)

# Copy embeddings + cls head (identical shapes)
student_model.bert.embeddings.load_state_dict(teacher_model.bert.embeddings.state_dict())
student_model.cls.load_state_dict(teacher_model.cls.state_dict())

# Copy every 2nd encoder layer: teacher layers 0,2,4,6,8,10 → student layers 0,1,2,3,4,5
for student_idx, teacher_idx in enumerate(range(0, teacher_config.num_hidden_layers, 2)):
    student_model.bert.encoder.layer[student_idx].load_state_dict(
        teacher_model.bert.encoder.layer[teacher_idx].state_dict()
    )

student_model = student_model.to(device)

del teacher_model, dataset_maker
torch.cuda.empty_cache()
print("Teacher model freed from GPU.")

student_params = sum(p.numel() for p in student_model.parameters())
print(f"Teacher: {teacher_config.num_hidden_layers} layers, {teacher_params / 1e6:.1f}M params")
print(f"Student: {student_config.num_hidden_layers} layers, {student_params / 1e6:.1f}M params (initialized from teacher)")
print(f"Compression: {student_params / teacher_params:.1%} of teacher size")

data_dir = os.path.join(os.getcwd(), "data", "batches")
batch_files = sorted([f for f in os.listdir(data_dir) if f.endswith(".pt")])
num_batches = len(batch_files)

optimizer = torch.optim.AdamW(student_model.parameters(), lr=5e-5)
kld_loss_fn = nn.KLDivLoss(reduction="batchmean")
ce_loss_fn = nn.CrossEntropyLoss()

T = 2.0
alpha = 0.5
epochs = 3

for epoch in range(epochs):
    student_model.train()
    epoch_loss = 0.0

    for i, batch_file in enumerate(batch_files):
        batch = torch.load(os.path.join(data_dir, batch_file), map_location=device)

        student_output = student_model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        student_logits = student_output.logits[batch["masked_indices"]]

        student_log_probs = F.log_softmax(student_logits / T, dim=-1)
        teacher_probs = F.softmax(batch["teachers_logits"] / T, dim=-1)
        distillation_loss = kld_loss_fn(student_log_probs, teacher_probs) * (T ** 2)

        hard_labels = batch["labels"][batch["masked_indices"]]
        hard_loss = ce_loss_fn(student_logits, hard_labels)

        loss = alpha * distillation_loss + (1 - alpha) * hard_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        del batch, student_output, student_logits
        torch.cuda.empty_cache()

        if (i + 1) % 10 == 0:
            print(f"\r  Epoch {epoch+1}/{epochs} | Batch {i+1}/{num_batches} | Loss: {loss.item():.4f}", end="", flush=True)

    avg_loss = epoch_loss / num_batches
    print(f"\r  Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}                    ")

print("\nTraining complete!")
student_model.save_pretrained(os.path.join(os.getcwd(), "data", "distilled_student_model"))
print("Student model saved.")

In [ ]:
model_path = os.path.join(os.getcwd(), "data", "distilled_student_model")
student_model = AutoModelForMaskedLM.from_pretrained(model_path).to(device)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
student_model.eval()

def predict_masked(text, top_k=5):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    mask_positions = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

    with torch.no_grad():
        logits = student_model(**inputs).logits

    for pos in mask_positions:
        probs = F.softmax(logits[0, pos], dim=-1)
        top_probs, top_ids = probs.topk(top_k)
        top_tokens = tokenizer.convert_ids_to_tokens(top_ids)

        context_token = tokenizer.convert_ids_to_tokens(inputs.input_ids[0, pos].unsqueeze(0))[0]
        print(f"\n[MASK] at token position {pos.item()} predictions:")
        for token, prob in zip(top_tokens, top_probs):
            print(f"  {token:15s} {prob.item():.4f}")

    best_ids = logits[0, mask_positions].argmax(dim=-1)
    filled = inputs.input_ids.clone()
    filled[0, mask_positions] = best_ids
    return tokenizer.decode(filled[0], skip_special_tokens=True)

test_sentences = [
    "The capital of France is located at [MASK].",
    "She went to the [MASK] to buy some groceries.",
    "Deep learning is a subset of [MASK] learning.",
    "Delhi is the capital of [MASK].",
    "The meaning of the life is [MASK].",
    "The [MASK] rises in the East [MASK].",
    "I am a [MASK] model."
]

for sentence in test_sentences:
    print(f"\nInput:  {sentence}")
    result = predict_masked(sentence)
    print(f"Output: {result}")